In [1]:
# ============================================================
# PIPELINE COMPLETO TFM (VERSIÓN FINAL CON LOGGING)
# ============================================================


# ============================================================
# A: PREPARACIÓN
# ============================================================

# ------------------------------------------------------------
# A1. Configuración e imports
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import GEOparse
import matplotlib.pyplot as plt
import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score, roc_curve

print("\n=== CONFIGURACIÓN ===")

RANDOM_STATE = 42
K_VALUES = [20, 50, 100, 500, 1000]


# ------------------------------------------------------------
# A2. Carga dataset GSE55348
# ------------------------------------------------------------

print("\n=== CARGANDO GSE55348 ===")

gse = GEOparse.get_GEO("GSE55348", destdir="../data")

expr_data = {}

for gsm_name, gsm in gse.gsms.items():
    expr_data[gsm_name] = gsm.table.set_index("ID_REF")["VALUE"]

X = pd.DataFrame(expr_data).T

print("Shape X:", X.shape)


# ------------------------------------------------------------
# A3. Labels
# ------------------------------------------------------------

print("\n=== EXTRACCIÓN DE LABELS ===")

y = []

for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", [])
    label = None

    for c in chars:
        c = c.lower()
        if "event:" in c:
            label = int(c.split(":")[-1].strip())

    if label is None:
        raise ValueError(f"No label en {gsm_name}")

    y.append(label)

y = np.array(y)

print("Distribución clases:", np.bincount(y))


# ------------------------------------------------------------
# A4. Log transform
# ------------------------------------------------------------

print("\n=== LOG TRANSFORM ===")

X_log = np.log2(X + 1)



07-Apr-2026 16:47:34 DEBUG utils - Directory ../data already exists. Skipping.
07-Apr-2026 16:47:34 INFO GEOparse - File already exist: using local version.
07-Apr-2026 16:47:34 INFO GEOparse - Parsing ../data\GSE55348_family.soft.gz: 
07-Apr-2026 16:47:34 DEBUG GEOparse - DATABASE: GeoMiame
07-Apr-2026 16:47:34 DEBUG GEOparse - SERIES: GSE55348
07-Apr-2026 16:47:34 DEBUG GEOparse - PLATFORM: GPL14951



=== CONFIGURACIÓN ===

=== CARGANDO GSE55348 ===


07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334487
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334488
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334489
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334490
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334491
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334492
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334493
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334494
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334495
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334496
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334497
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334498
07-Apr-2026 16:47:35 DEBUG GEOparse - SAMPLE: GSM1334499
07-Apr-2026 16:47:36 DEBUG GEOparse - SAMPLE: GSM1334500
07-Apr-2026 16:47:36 DEBUG GEOparse - SAMPLE: GSM1334501
07-Apr-2026 16:47:36 DEBUG GEOparse - SAMPLE: GSM1334502
07-Apr-2026 16:47:36 DEBUG GEOparse - SAMPLE: GSM1334503
07-Apr-2026 16:47:36 DEBUG GEOp

Shape X: (53, 29377)

=== EXTRACCIÓN DE LABELS ===
Distribución clases: [30 23]

=== LOG TRANSFORM ===


In [2]:
# ============================================================
# B: EXPERIMENTACIÓN COMPLETA (MÉTRICAS + CONFUSIÓN)
# ============================================================

from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    balanced_accuracy_score,
    matthews_corrcoef
)

print("\n=== EXPERIMENTO K ===")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

all_results = []

for K in K_VALUES:

    print(f"\nEvaluando K = {K}")

    pipelines = {

        # ------------------------------------------------------------
        # BASELINE
        # ------------------------------------------------------------
        "LR baseline": Pipeline([
            ("variance", VarianceThreshold(0.01)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000))
        ]),

        # ------------------------------------------------------------
        # ANOVA
        # ------------------------------------------------------------
        "LR + ANOVA": Pipeline([
            ("variance", VarianceThreshold(0.01)),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=K)),
            ("model", LogisticRegression(max_iter=1000))
        ]),

        # ------------------------------------------------------------
        # MUTUAL INFORMATION
        # ------------------------------------------------------------
        "LR + MutualInfo": Pipeline([
            ("variance", VarianceThreshold(0.01)),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(lambda X, y: mutual_info_classif(X, y), k=K)),
            ("model", LogisticRegression(max_iter=1000))
        ]),

        # ------------------------------------------------------------
        # RFE
        # ------------------------------------------------------------
        "LR + RFE": Pipeline([
            ("variance", VarianceThreshold(0.01)),
            ("scaler", StandardScaler()),
            ("select", RFE(LogisticRegression(max_iter=1000), n_features_to_select=K, step=50)),
            ("model", LogisticRegression(max_iter=1000))
        ])
    }

    for name, pipe in pipelines.items():

        print(f"Modelo: {name}")
        start = time.time()

        # ------------------------------------------------------------
        # CV CLÁSICO (AUC, F1, ACC)
        # ------------------------------------------------------------
        scores = cross_validate(
            pipe,
            X_log,
            y,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "f1": "f1",
                "accuracy": "accuracy"
            },
            n_jobs=2
        )

        # ------------------------------------------------------------
        # PREDICCIONES OUT-OF-FOLD (CLAVE)
        # ------------------------------------------------------------
        y_pred = cross_val_predict(pipe, X_log, y, cv=cv, n_jobs=2)
        y_prob = cross_val_predict(pipe, X_log, y, cv=cv, method="predict_proba", n_jobs=2)[:, 1]

        # ------------------------------------------------------------
        # MATRIZ DE CONFUSIÓN GLOBAL
        # ------------------------------------------------------------
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

        # ------------------------------------------------------------
        # MÉTRICAS CLÍNICAS
        # ------------------------------------------------------------
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall positivo
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = precision_score(y, y_pred)
        recall = recall_score(y, y_pred)

        balanced_acc = balanced_accuracy_score(y, y_pred)
        mcc = matthews_corrcoef(y, y_pred)

        end = time.time()
        print(f"Tiempo: {end - start:.2f} s")

        # ------------------------------------------------------------
        # GUARDADO DE RESULTADOS
        # ------------------------------------------------------------
        all_results.append({
            "Model": name,
            "K": K,

            # Métricas CV estándar
            "ROC-AUC": np.mean(scores["test_roc_auc"]),
            "F1": np.mean(scores["test_f1"]),
            "Accuracy": np.mean(scores["test_accuracy"]),

            # Métricas clínicas
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "Balanced Acc": balanced_acc,
            "MCC": mcc,

            # Confusion matrix
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn
        })


# ============================================================
# B2: L1 (REGULARIZACIÓN)
# ============================================================

print("\n=== EXPERIMENTO L1 ===")

C_VALUES = [0.01, 0.1, 1]
l1_results = []

for C in C_VALUES:

    print(f"Evaluando C = {C}")

    pipe = Pipeline([
        ("variance", VarianceThreshold(0.01)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1",
            solver="saga",
            C=C,
            max_iter=5000
        ))
    ])

    # ------------------------------------------------------------
    # CV estándar
    # ------------------------------------------------------------
    scores = cross_validate(
        pipe,
        X_log,
        y,
        cv=cv,
        scoring={
            "roc_auc": "roc_auc",
            "f1": "f1",
            "accuracy": "accuracy"
        },
        n_jobs=2
    )

    # ------------------------------------------------------------
    # PREDICCIONES OOF
    # ------------------------------------------------------------
    y_pred = cross_val_predict(pipe, X_log, y, cv=cv, n_jobs=2)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = precision_score(y, y_pred)

    balanced_acc = balanced_accuracy_score(y, y_pred)
    mcc = matthews_corrcoef(y, y_pred)

    l1_results.append({
        "Model": "LR L1",
        "C": C,

        "ROC-AUC": np.mean(scores["test_roc_auc"]),
        "F1": np.mean(scores["test_f1"]),
        "Accuracy": np.mean(scores["test_accuracy"]),

        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "Balanced Acc": balanced_acc,
        "MCC": mcc,

        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })


=== EXPERIMENTO K ===

Evaluando K = 20
Modelo: LR baseline
Tiempo: 6.30 s
Modelo: LR + ANOVA
Tiempo: 4.28 s
Modelo: LR + MutualInfo


KeyboardInterrupt: 

In [ ]:

# ============================================================
# C: ENTRENAMIENTO FINAL Y SELECCIÓN DE GENES
# ============================================================

# ------------------------------------------------------------
# C1. Entrenamiento modelo L1 final
# Entrenamos en TODO el dataset para obtener biomarcadores
# ------------------------------------------------------------

print("\n=== ENTRENAMIENTO MODELO FINAL ===")

l1_pipe = Pipeline([
    ("variance", VarianceThreshold(0.01)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=0.1,
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

# Entrenamos el modelo con todos los datos
l1_pipe.fit(X_log, y)

# Extraemos coeficientes del modelo
coefs = l1_pipe.named_steps["model"].coef_[0]

# Recuperamos qué genes han pasado el filtro de varianza
mask = l1_pipe.named_steps["variance"].get_support()
genes_after_variance = X_log.columns[mask]

# Seleccionamos genes con coeficiente distinto de 0
selected_genes = genes_after_variance[coefs != 0]

print("Número de genes seleccionados (probes):", len(selected_genes))
print(selected_genes)


# ------------------------------------------------------------
# C2. MAPPING A GENES REALES (MUY IMPORTANTE)
# Convertimos probes → símbolos génicos
# ------------------------------------------------------------

print("\n=== MAPEO A GENES (GSE55348) ===")

# Cargamos anotación de la plataforma Illumina
platform_55348 = gse.gpls[list(gse.gpls.keys())[0]]
annot_55348 = platform_55348.table

# Creamos diccionario probe → símbolo génico
map_55348 = dict(zip(annot_55348['ID'], annot_55348['Symbol']))

# Aplicamos mapping a los genes seleccionados
mapped_selected_genes = [map_55348.get(g) for g in selected_genes]

# Eliminamos valores None y duplicados (manteniendo orden)
mapped_selected_genes = list(dict.fromkeys(
    [g for g in mapped_selected_genes if g is not None]
))

print("Número de genes mapeados (únicos):", len(mapped_selected_genes))
print(mapped_selected_genes)



In [ ]:

# ============================================================
# D: VALIDACIÓN EXTERNA (CORREGIDO Y ROBUSTO)
# ============================================================

print("\n=== VALIDACIÓN EXTERNA ===")

gse_58984 = GEOparse.get_GEO("GSE58984", destdir="../data")

# ------------------------------------------------------------
# D1. Cargar datos
# ------------------------------------------------------------

expr_data_58984 = {}
for gsm_name, gsm in gse_58984.gsms.items():
    expr_data_58984[gsm_name] = gsm.table.set_index("ID_REF")["VALUE"]

X_58984 = pd.DataFrame(expr_data_58984).T
X_58984_log = np.log2(X_58984 + 1)

# ------------------------------------------------------------
# D2. Labels
# ------------------------------------------------------------

y_58984 = []
for gsm_name in X_58984.index:
    gsm = gse_58984.gsms[gsm_name]
    chars = gsm.metadata.get("characteristics_ch1", [])

    for c in chars:
        if "distant metastasis" in c.lower():
            y_58984.append(int(c.split(":")[-1].strip()))

y_58984 = np.array(y_58984)

print("Distribución externa:", np.bincount(y_58984))


# ------------------------------------------------------------
# D3. Mapping a genes (igual que antes)
# ------------------------------------------------------------

print("\n=== MAPEO A GENES (GSE58984) ===")

platform_58984 = gse_58984.gpls[list(gse_58984.gpls.keys())[0]]
annot_58984 = platform_58984.table

map_58984 = dict(zip(annot_58984['ID'], annot_58984['Gene Symbol']))

X_58984_genes = X_58984_log.rename(columns=map_58984)
# Limpiar nombres
X_58984_genes.columns = [
    str(g).split("///")[0].strip() if g is not None else None
    for g in X_58984_genes.columns
]

# Eliminar columnas sin nombre
X_58984_genes = X_58984_genes.loc[:, X_58984_genes.columns.notna()]

# AGRUPAR genes duplicados (igual que en train)
X_58984_genes = X_58984_genes.T.groupby(level=0).mean().T
# LIMPIEZA de nombres (CLAVE)
X_58984_genes.columns = [
    str(g).split("///")[0].strip() if g is not None else None
    for g in X_58984_genes.columns
]

# Eliminamos columnas sin nombre válido
X_58984_genes = X_58984_genes.loc[:, X_58984_genes.columns.notna()]


# ------------------------------------------------------------
# D4. También limpiamos los genes del training
# ------------------------------------------------------------

mapped_selected_genes_clean = [
    g.split("///")[0].strip() if isinstance(g, str) else g
    for g in mapped_selected_genes
]


# ------------------------------------------------------------
# D5. Intersección REAL
# ------------------------------------------------------------

common_genes = [
    g for g in mapped_selected_genes_clean
    if g in X_58984_genes.columns
]

print("Genes comunes:", len(common_genes))
print(common_genes)


# ------------------------------------------------------------
# D6. CONTROL (MUY IMPORTANTE)
# ------------------------------------------------------------

if len(common_genes) == 0:
    raise ValueError(
        "No hay genes comunes entre datasets. "
        "Esto probablemente se debe a diferencias de anotación entre plataformas."
    )


# ------------------------------------------------------------
# D7. Entrenamiento y evaluación
# ------------------------------------------------------------

# 1. Nos quedamos SOLO con los probes seleccionados
X_train = X_log[selected_genes].copy()

# 2. Mapping probe → gen
probe_to_gene = dict(zip(selected_genes, mapped_selected_genes_clean))

# 3. Renombrar
X_train = X_train.rename(columns=probe_to_gene)

# 4. Agrupar genes duplicados
X_train = X_train.T.groupby(level=0).mean().T

# 5. Seleccionar genes comunes
X_train = X_train[common_genes]

# ------------------------------------------------------------
# IMPORTANTE: asegurar mismo orden en test
# ------------------------------------------------------------

# Nos aseguramos de que test tiene SOLO esos genes
X_test = X_58984_genes.copy()

# Nos quedamos con los genes comunes (y en mismo orden)
X_test = X_test.reindex(columns=common_genes)

# ------------------------------------------------------------
# DEBUG
# ------------------------------------------------------------

print("Train columns:", list(X_train.columns))
print("Test columns:", list(X_test.columns))

print("Shape train:", X_train.shape)
print("Shape test:", X_test.shape)

# ------------------------------------------------------------
# Modelo
# ------------------------------------------------------------

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

final_model.fit(X_train, y)

# Predicción probabilística
y_pred = final_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# MÉTRICAS
# ------------------------------------------------------------

from sklearn.metrics import confusion_matrix

# AUC
auc = roc_auc_score(y_58984, y_pred)
auc_inv = roc_auc_score(y_58984, 1 - y_pred)

# Convertimos probabilidades a clases (umbral 0.5)
y_pred_class = (y_pred >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_58984, y_pred_class).ravel()

sensibilidad = tp / (tp + fn) if (tp + fn) > 0 else 0
especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("\n=== MÉTRICAS EXTERNAS ===")
print("AUC:", auc)
print("AUC invertido:", auc_inv)
print("Sensibilidad:", sensibilidad)
print("Especificidad:", especificidad)



In [ ]:

# ============================================================
# E: VISUALIZACIÓN
# ============================================================

print("\n=== CURVA ROC ===")

fpr, tpr, _ = roc_curve(y_58984, y_pred)
fpr_inv, tpr_inv, _ = roc_curve(y_58984, 1 - y_pred)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.3f})")
plt.plot(fpr_inv, tpr_inv, linestyle="--", label=f"Invertida (AUC={auc_inv:.3f})")
plt.plot([0,1],[0,1], linestyle=":")
plt.legend()
plt.grid()
plt.show()

print("\n=== FIN PIPELINE ===")